# 001 — Data Chunking Strategies
**Ingestion Series** | The foundation most RAG pipelines get wrong

Covers: Fixed-size · Recursive character · Semantic · Late chunking


In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── 1. Fixed-Size Chunking ───────────────────────────────────────────────
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

fixed = CharacterTextSplitter(chunk_size=500, chunk_overlap=50, separator="")
c_fixed = fixed.split_text(ai_text[:6000])
print(f"Fixed chunks: {len(c_fixed)}")
sizes = [len(c) for c in c_fixed]
print(f"  min={min(sizes)}  max={max(sizes)}  avg={sum(sizes)//len(sizes)}")
print("\nSample — notice potential mid-word cuts:")
print(repr(c_fixed[2][:200]))


In [ ]:
# ── 2. Recursive Character Splitting ────────────────────────────────────
rec = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""]
)
c_rec = rec.split_text(ai_text[:6000])
print(f"Recursive chunks: {len(c_rec)}")
sizes = [len(c) for c in c_rec]
print(f"  min={min(sizes)}  max={max(sizes)}  avg={sum(sizes)//len(sizes)}")
print("\nSample — clean sentence boundary:")
print(c_rec[2][:300])


In [ ]:
# ── 3. Semantic Chunking (embedding-distance based) ──────────────────────
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def semantic_split(text, threshold=0.78, max_sentences=35):
    sents = [s.strip() for s in text.replace("\n", " ").split(". ")
             if len(s.strip()) > 30][:max_sentences]
    if len(sents) < 2:
        return [text]
    vecs = embeddings.embed_documents(sents)
    chunks, cur = [], [sents[0]]
    for i in range(1, len(sents)):
        if cosine(vecs[i-1], vecs[i]) < threshold and len(" ".join(cur)) > 100:
            chunks.append(". ".join(cur) + ".")
            cur = [sents[i]]
        else:
            cur.append(sents[i])
    if cur:
        chunks.append(". ".join(cur) + ".")
    return chunks

c_sem = semantic_split(ai_text[:4000])
print(f"Semantic chunks: {len(c_sem)}")
for i, c in enumerate(c_sem[:3]):
    print(f"\nChunk {i+1} ({len(c)} chars): {c[:120]}...")


In [ ]:
# ── 4. Late Chunking (context-enriched embeddings) ───────────────────────
def late_chunk(text, chunk_size=400, overlap=50, alpha=0.7):
    '''Blend each chunk embedding with the full-doc embedding for richer context.'''
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    chunks = splitter.split_text(text)
    doc_vec = np.array(embeddings.embed_query(text[:8000]))
    chunk_vecs = embeddings.embed_documents(chunks)
    enriched = []
    for cv in chunk_vecs:
        blended = alpha * np.array(cv) + (1 - alpha) * doc_vec
        enriched.append((blended / np.linalg.norm(blended)).tolist())
    return chunks, enriched

lc_chunks, lc_vecs = late_chunk(ai_text[:4000])
print(f"Late chunks: {len(lc_chunks)}  |  embedding dim: {len(lc_vecs[0])}")
print("\nEach embedding is 70% chunk + 30% document context.")
print("Retrieval stays accurate even for out-of-context chunks.")


In [ ]:
# ── Strategy Comparison ──────────────────────────────────────────────────
import pandas as pd, time

sample = ai_text[:6000]

t0 = time.time(); n_fixed   = len(fixed.split_text(sample));       t_fixed   = time.time()-t0
t0 = time.time(); n_rec     = len(rec.split_text(sample));         t_rec     = time.time()-t0
t0 = time.time(); n_sem     = len(semantic_split(sample));         t_sem     = time.time()-t0
t0 = time.time(); n_lc      = len(late_chunk(sample[:3000])[0]);   t_lc      = time.time()-t0

df = pd.DataFrame({
    "Strategy":      ["Fixed-size", "Recursive", "Semantic", "Late Chunking"],
    "Chunks":        [n_fixed, n_rec, n_sem, n_lc],
    "Boundary":      ["None", "Structure", "Meaning", "Meaning"],
    "Context Qual":  ["Low", "Medium", "High", "Very High"],
    "Build Time":    [f"{t_fixed*1000:.0f}ms", f"{t_rec*1000:.0f}ms",
                      f"{t_sem:.2f}s", f"{t_lc:.2f}s"],
    "Best For":      ["Code/tables", "General text", "Long articles", "Precision Q&A"],
})
pd.set_option('display.max_colwidth', 20)
pd.set_option('display.width', 120)
print(df.to_string(index=False))
print("\n✓ Semantic & Late chunking have embed cost; Recursive is the best default.")


## Key Takeaways
| Strategy | Default choice when… |
|---|---|
| Recursive | Always start here (chunk_size=512, overlap=50) |
| Semantic | Topics shift within paragraphs |
| Late chunking | Single chunk out of context loses meaning |
